## Tải các package cần thiết và load dữ liệu

In [1]:
!pip install "spacy==3.7.4" "scispacy==0.5.4"
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Using cached spacy-3.7.4-cp311-cp311-win_amd64.whl.metadata (27 kB)
  Using cached scispacy-0.5.4-py3-none-any.whl.metadata (16 kB)
  Using cached weasel-0.3.4-py3-none-any.whl.metadata (4.7 kB)
  Using cached typer-0.9.4-py3-none-any.whl.metadata (14 kB)
Using cached spacy-3.7.4-cp311-cp311-win_amd64.whl (12.1 MB)
Using cached scispacy-0.5.4-py3-none-any.whl (45 kB)
Using cached typer-0.9.4-py3-none-any.whl (45 kB)
Using cached weasel-0.3.4-py3-none-any.whl (50 kB)

  Attempting uninstall: typer

    Found existing installation: typer 0.24.1

    Uninstalling typer-0.24.1:

      Successfully uninstalled typer-0.24.1

   ---------------------------------------- 0/4 [typer]
   ---------------------------------------- 0/4 [typer]
   ---------------------------------------- 0/4 [typer]
  Attempting uninstall: weasel
   ---------------------------------------- 0/4 [typer]
    Found existing installation: weasel 0.4.3
   ---------------------------------------- 0/4 [typer]
   ---------- 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi-cli 0.0.16 requires typer>=0.15.1, but you have typer 0.9.4 which is incompatible.
fastapi-cloud-cli 0.6.0 requires typer>=0.12.3, but you have typer 0.9.4 which is incompatible.
gradio 6.1.0 requires pydantic<=2.12.4,>=2.11.10, but you have pydantic 2.12.5 which is incompatible.
gradio 6.1.0 requires typer<1.0,>=0.12, but you have typer 0.9.4 which is incompatible.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.9.4 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\ACER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz (14.8 MB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\ACER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [130]:
import spacy
import scispacy
import nltk
import json
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [131]:
corpus_path = './Data/corpus.jsonl'
queries_path = './Data/queries.jsonl'

In [132]:
def read_json(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [133]:
corpus_data = read_json(corpus_path)
print("Số tài liệu: ", len(corpus_data))
queries_data = read_json(queries_path)
print("Số câu truy vấn: ", len(queries_data))

Số tài liệu:  3633
Số câu truy vấn:  3237


In [134]:
corpus_data[:5]

[{'_id': 'MED-10',
  'title': 'Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland',
  'text': 'Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.08–9.0 ye

In [135]:
queries_data[:5]

[{'_id': 'PLAIN-3',
  'text': 'Breast Cancer Cells Feed on Cholesterol',
  'metadata': {'url': 'http://nutritionfacts.org/2015/07/14/breast-cancer-cells-feed-on-cholesterol/'}},
 {'_id': 'PLAIN-4',
  'text': 'Using Diet to Treat Asthma and Eczema',
  'metadata': {'url': 'http://nutritionfacts.org/2015/07/09/using-diet-to-treat-asthma-and-eczema/'}},
 {'_id': 'PLAIN-5',
  'text': 'Treating Asthma With Plants vs. Pills',
  'metadata': {'url': 'http://nutritionfacts.org/2015/07/07/treating-asthma-with-plants-vs-pills/'}},
 {'_id': 'PLAIN-6',
  'text': 'How Fruits and Vegetables Can Treat Asthma',
  'metadata': {'url': 'http://nutritionfacts.org/2015/07/02/how-fruits-and-vegetables-can-treat-asthma/'}},
 {'_id': 'PLAIN-7',
  'text': 'How Fruits and Vegetables Can Prevent Asthma',
  'metadata': {'url': 'http://nutritionfacts.org/2015/06/30/how-fruits-and-vegetables-can-prevent-asthma/'}}]

## Tiền xử lý

Các bước tiền xử lý bao gồm:
- 1. Chuyển thành chữ viết thường.
- 2. Tách entity bằng thư viện `Scispacy`.
- 3. Tách Noun, verb, adj bằng `Scispacy`.
- 4. Áp dụng word lemmatization.
- 5. Loại stopwords, punct và number bằng `nltk`. 
- 6. Lưu lại thành văn bản với các term ngăn cách bởi khoảng trắng.

In [136]:
nlp = spacy.load("en_core_sci_sm")
STOPWORDS = set(stopwords.words('english'))

In [138]:
def sparse_phrase(phrase):
    tokens = [
            tok.lemma_ for tok in phrase
            if not tok.is_punct
            and tok.lemma_ not in STOPWORDS
        ]
    term = "_".join(tokens) if tokens else None
    return term

In [166]:
def _preprocessing(text):
    # Bước 1: chuyển thành chữ thường
    text = text.lower()
    # Xử lý với thư viện scispacy
    doc = nlp(text)
    terms = []
    
    # Bước 2: Lấy entities
    entity_spans = set()
    for ent in doc.ents: #doc.ents là danh sách ents
        entity_spans.update(range(ent.start, ent.end))

        # Bước 4 + 5
        term = sparse_phrase(ent) 
        if term:
            terms.append(term)
            
    # Bước 3: Lấy Noun, adj, verb
    for i, tok in enumerate(doc):
        if i not in entity_spans:
            is_content_word = tok.pos_ in ("NOUN", "ADJ", "VERB")
            
            # Bước 4 + 5
            if (is_content_word) and not tok.is_punct and tok.lemma_ not in STOPWORDS:
                terms.append(tok.lemma_)

    text = " ".join(terms)
    return text

In [167]:
text = [
    "Aspirin reduces platelet aggregation in cardiovascular disease",
    "Potassium intake can help reduce high blood pressure",
    "the current study shows promising results in adult patients",
    "Olive Oil and Artery Function",
    "How Doctors Responded to Being Named a Leading Killer",
    "trans fats",
    "4-nonylphenol (NP) and 4-octylphenol (OP) in 59 human milk samples",
    "14 years old"
]

In [168]:
for t in text:
    print(_preprocessing(t))

aspirin platelet_aggregation cardiovascular_disease reduce
potassium_intake high_blood_pressure help reduce
study adult patient current show promising result
olive_oil artery_function
doctor respond name lead killer
trans_fat
4-nonylphenol 4-octylphenol human np op milk sample
year old


In [153]:
def preprocessing(data, is_corpus = 1):
    cleaned_data = []
    cnt = 0
    
    for d in data:
        record = {}
        record["_id"] = d["_id"]
        record["metadata"] = d["metadata"]
        if is_corpus:
            record["title"] = d["title"]
        record["text"] = _preprocessing(d["text"])
        if record["_id"] == '':
            print("Nội dung trống, không tìm được entity và phrase trong văn bản: ")
            print(f'ID: {record["_id"]}, Nội dung gốc: {d["text"]}')
            cnt += 1
            continue
        cleaned_data.append(record)
        
    print(f"Có {cnt} văn bản trống sau khi preprocess")
    return cleaned_data

In [155]:
clean_corpus = preprocessing(corpus_data)

Có 0 văn bản trống sau khi preprocess


In [169]:
corpus_data[0]["text"], clean_corpus[0]["text"]

('Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.08–9.0 years) 6,011 participants died, of which 3,619 (60.2%) was due to breast cancer. After adjustment for age, tumor character

In [170]:
clean_queries = preprocessing(queries_data, is_corpus=0)

Có 0 văn bản trống sau khi preprocess


In [172]:
queries_data[0]["text"], clean_queries[0]["text"]

('Breast Cancer Cells Feed on Cholesterol',
 'breast_cancer cholesterol cell feed')

In [175]:
# Bước 6: lưu xuống file đã tiền xử lý
def save_jsonl(data: list[dict], path: str):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print("Đã lưu thành công xuống: ", path)

In [176]:
clean_corpus_path = "./Data/clean_corpus.jsonl"
clean_queries_path = "./Data/clean_queries.jsonl"

save_jsonl(clean_corpus, clean_corpus_path)
save_jsonl(clean_queries, clean_queries_path)

Đã lưu thành công xuống:  ./Data/clean_corpus.jsonl
Đã lưu thành công xuống:  ./Data/clean_queries.jsonl


## Report

Pipeline dự tính ban đầu: lower -> entity & phrase noun -> UMLS linking cho entity -> Stopwords & Lemmatization -> Save.  
Vấn đề:
- Các entity và phrase noun gần như trùng nhau toàn bộ -> bỏ phrase noun.
- Các thuật ngữ khoa học UMLS vô cùng hàn lâm, kể cả cho corpus khiến chỉ mục trở nên phức tạp -> bỏ UMLS.
- Bỏ qua các từ có ý nghĩa quan trọng với câu như danh từ, động từ và tính từ -> Lấy lại NOUN, VERB, ADJ.
- Các con số trong hệ thống IR cho y tế không thực sự cần thiết, ví dụ như các con số thể hiện kết quả tính toán, nghiên cứu. Khi truy vấn thông tin, đa số người dùng không cần tìm chính xác các con số này -> Bỏ number.

Pipeline thực tế: lower -> entity, VERB, NOUN, ADJ -> Lọc Stopwords & Lemmatization & Punct & Number-> Save.  

Hiện tại, chỉ dùng text trong queries và text trong corpus, bỏ qua title của corpus.